# Project 19 — Mechanistic Model: 1-Compartment PK (Bayesian ODE)

**Scenario.** A drug given as a single IV bolus is eliminated from a single compartment by a first-order process: $dC/dt = -kC$, $C(0)=D/V$. We observe noisy concentrations and infer the **mechanistic rate constants** $k$ (elimination) and $V$ (volume).

**New skill.** ODE inference, identifiability, and the compute cost of solvers. **Key pitfall.** *Practical non-identifiability*: if the design doesn't pin down both $C_0=D/V$ (early times) and the decay slope $k$ (late times), $k$ and $V$ correlate — the PK analogue of the Michaelis-Menten $V_\max/K_m$ correlation.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
az.style.use('arviz-darkgrid')
RNG = 20240601

## Compute note (read first)

This model has a **closed-form solution** $C(t)=(D/V)e^{-kt}$. We fit that analytic form (exact, fast). PyMC also offers `pymc.ode.DifferentialEquation`, which integrates the ODE numerically at every sampler step — the genuine ODE-inference workflow, but far slower. We show it in Step 7 with tiny settings. **For a system with a closed form, using it is the responsible choice**; reserve the numerical solver for ODEs without one.

## Step 1 — Problem & data-generating story

$y_i = C(t_i) + \varepsilon_i$, $\varepsilon_i\sim N(0,\sigma^2)$, with $C(t)=(D/V)e^{-kt}$. **Assumptions:** (a) one well-mixed compartment; (b) first-order (linear) elimination; (c) instantaneous IV bolus at $t=0$; (d) additive Gaussian measurement noise. Truths: $k=0.35$/h, $V=8$ L, $\sigma=0.4$ mg/L. The design includes **early** points (pin $C_0=D/V$) and **late** points (pin the slope $k$).

In [ ]:
from data.generate_data import generate, analytic_C
data = generate()
print(f"n={data['n']} timepoints, dose={data['dose']:.0f} mg")
print(f"truth: k={data['truth']['k']}, V={data['truth']['V']}, sigma={data['truth']['sigma']}")
tt = np.linspace(0, data['t'].max(), 200)
fig, ax = plt.subplots(figsize=(6.5,3.6))
ax.plot(tt, analytic_C(tt), 'k--', label='true C(t)')
ax.scatter(data['t'], data['y'], color='#4C72B0', zorder=3, label='noisy data')
ax.set(xlabel='time (h)', ylabel='concentration (mg/L)', title='PK profile')
ax.legend(); plt.tight_layout()

## Step 2 — Model specification (mechanism + justified priors)

We sample $k$ and $V$ on the **log scale** (they are positive and span orders of magnitude). Priors: $\log k\sim N(-1,0.7)$ (so $k\approx 0.37$/h), $\log V\sim N(2,0.5)$ (so $V\approx 7.4$ L), $\sigma\sim\text{HalfNormal}(1)$. These encode plausible physiological ranges — mechanistic models *should* use informed priors.

In [ ]:
from model import build_model, fit, fit_ode
model = build_model(data)
model

## Step 3 — Prior predictive checks

We simulate concentration curves implied by the priors. They should look like plausible decay profiles — right order of magnitude for $C_0$, sensible half-lives — not absurd (megagram concentrations or instantaneous disappearance).

In [ ]:
rng = np.random.default_rng(RNG)
fig, ax = plt.subplots(figsize=(6.5,3.6))
for _ in range(12):
    k = np.exp(rng.normal(-1,0.7)); V = np.exp(rng.normal(2,0.5))
    ax.plot(tt, (data['dose']/V)*np.exp(-k*tt), lw=1, alpha=0.7)
ax.set(xlabel='time (h)', ylabel='C(t)', title='Prior predictive PK curves')
plt.tight_layout()

## Step 4 — Inference (NUTS)

Settings: `draws=600, tune=1000, chains=2, target_accept=0.9, cores=1`. The log parameterisation gives a well-behaved geometry; the analytic likelihood makes this fast.

In [ ]:
idata = fit(data, draws=600, tune=1000, chains=2, seed=101)

## Step 5 — Computational diagnostics & identifiability

Check $\hat R$, ESS, divergences. The key plot is the **$(k, V)$ pair plot**: with the full design it is a compact blob; a strong diagonal correlation would warn of practical non-identifiability. We also look at clearance $CL=k\cdot V$, which is often *better* identified than either factor.

In [ ]:
print(az.summary(idata, var_names=['k','V','sigma']))
print('divergences:', int(idata.sample_stats['diverging'].sum()))

In [ ]:
az.plot_pair(idata, var_names=['k','V'], kind='scatter',
             scatter_kwargs={'alpha':0.2}); plt.tight_layout()

In [ ]:
k = idata.posterior['k'].values.ravel(); V = idata.posterior['V'].values.ravel()
CL = k*V
print(f'corr(k,V) = {np.corrcoef(k,V)[0,1]:.3f}')
print(f'clearance CL=k*V: mean={CL.mean():.2f} L/h '
      f'(94% [{np.percentile(CL,3):.2f},{np.percentile(CL,97):.2f}])')

## Step 6 — Posterior predictive checks

Overlay posterior-predictive concentration curves on the data; the observed points should sit inside the predictive band. We also report a Bayesian p-value on a discrepancy (e.g. the residual sum of squares).

In [ ]:
ax = az.plot_ppc(idata, num_pp_samples=100); plt.tight_layout()

## Step 7 — Model criticism & the genuine ODE solver

We demonstrate the same model via `pymc.ode.DifferentialEquation` (numerical integration) on tiny settings, and confirm it agrees with the analytic fit. This is the workflow you would use for an ODE **without** a closed form (e.g. saturable Michaelis-Menten elimination). It is much slower — note the time.

In [ ]:
import time; t0=time.time()
idata_ode = fit_ode(data, draws=100, tune=200, chains=2, seed=101)
print(f'ODE-solver fit took {time.time()-t0:.1f}s (vs the analytic fit, ~instant)')
print(az.summary(idata_ode, var_names=['k','V']))
print('analytic k,V:', float(idata.posterior['k'].mean()), float(idata.posterior['V'].mean()))

## Step 8 — Decision & communication

Translate into a dosing-relevant quantity: the **half-life** $t_{1/2}=\ln 2/k$ and the **clearance** $CL=kV$, each with a credible interval — the numbers a pharmacologist uses to choose a dosing interval.

In [ ]:
thalf = np.log(2)/k
print(f'half-life t1/2 = {thalf.mean():.2f} h '
      f'(94% [{np.percentile(thalf,3):.2f}, {np.percentile(thalf,97):.2f}])')
print(f'clearance CL = {CL.mean():.2f} L/h '
      f'(94% [{np.percentile(CL,3):.2f}, {np.percentile(CL,97):.2f}])')

**Conclusion (for a collaborator).** The mechanism is identified by this design: $k$ and $V$ recover their true values, and clearance/half-life come with usable intervals. The identifiability hinged on having **both** early and late samples — see the broken notebook for what happens without the early points.